<a href="https://colab.research.google.com/github/6hamuge/Beyond-ETRI/blob/main/usage_status%2C_wifi%2C_hr_feature.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## 0. 환경 설정 및 패키지 설치

In [ ]:
# Google Colab 환경 확인
import sys
IN_COLAB = 'google.colab' in sys.modules
print(f'Colab 환경: {IN_COLAB}')

# 필수 패키지 설치
!pip install pyarrow pandas numpy scikit-learn torch -q
!pip install matplotlib seaborn tqdm -q

print('✅ 패키지 설치 완료')

Colab 환경: True
✅ 패키지 설치 완료


## 1. 데이터 업로드

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DATA_DIR = '/content/drive/MyDrive/ch2025/'
print(f'✅ Drive 마운트 완료. DATA_DIR = {DATA_DIR}')

Mounted at /content/drive
✅ Drive 마운트 완료. DATA_DIR = /content/drive/MyDrive/ch2025/


In [ ]:
import pandas as pd
import numpy as np
import os
import warnings
warnings.filterwarnings('ignore')

FILE_MAP = {
    'ac_status'   : 'ch2025_mACStatus.parquet',
    'activity'    : 'ch2025_mActivity.parquet',
    'ambience'    : 'ch2025_mAmbience.parquet',
    'ble'         : 'ch2025_mBle.parquet',
    'gps'         : 'ch2025_mGps.parquet',
    'light'       : 'ch2025_mLight.parquet',
    'screen'      : 'ch2025_mScreenStatus.parquet',
    'usage_stats' : 'ch2025_mUsageStats.parquet',
    'wifi'        : 'ch2025_mWifi.parquet',
    'hr'          : 'ch2025_wHr.parquet',
    'w_light'     : 'ch2025_wLight.parquet',
    'pedo'        : 'ch2025_wPedo.parquet',
}

# ─── 데이터 로드 ───────────────────────────────────────────
raw = {}
for key, fname in FILE_MAP.items():
    path = os.path.join(DATA_DIR, "ch2025_data_items/", fname)
    if os.path.exists(path):
        raw[key] = pd.read_parquet(path)
        print(f'✅ {key:12s}: {raw[key].shape}  | cols: {list(raw[key].columns)}')
    else:
        print(f'❌ {key}: 파일 없음 ({path})')

print(f'\n📊 로드된 테이블: {len(raw)}개')

✅ ac_status   : (939896, 3)  | cols: ['subject_id', 'timestamp', 'm_charging']
✅ activity    : (961062, 3)  | cols: ['subject_id', 'timestamp', 'm_activity']
✅ ambience    : (476577, 3)  | cols: ['subject_id', 'timestamp', 'm_ambience']
✅ ble         : (21830, 3)  | cols: ['subject_id', 'timestamp', 'm_ble']
✅ gps         : (800611, 3)  | cols: ['subject_id', 'timestamp', 'm_gps']
✅ light       : (96258, 3)  | cols: ['subject_id', 'timestamp', 'm_light']
✅ screen      : (939653, 3)  | cols: ['subject_id', 'timestamp', 'm_screen_use']
✅ usage_stats : (45197, 3)  | cols: ['subject_id', 'timestamp', 'm_usage_stats']
✅ wifi        : (76336, 3)  | cols: ['subject_id', 'timestamp', 'm_wifi']
✅ hr          : (382918, 3)  | cols: ['subject_id', 'timestamp', 'heart_rate']
✅ w_light     : (633741, 3)  | cols: ['subject_id', 'timestamp', 'w_light']
✅ pedo        : (748100, 9)  | cols: ['subject_id', 'timestamp', 'step', 'step_frequency', 'running_step', 'walking_step', 'distance', 'speed', 'burne

## 4. 피처 엔지니어링 (일별 집계)

각 센서의 원시 시계열 → **하루 단위 통계 특징**으로 변환  
핵심: `subject_id + date` 기준 집계

In [ ]:
# ─── 유틸리티 함수 ────────────────────────────────────────

def get_timestamp_col(df):
    """타임스탬프 컬럼 자동 탐지"""
    candidates = ['timestamp', 'time', 'datetime', 'ts', 'date']
    for c in df.columns:
        if any(k in c.lower() for k in candidates):
            return c
    # datetime dtype으로 탐지
    for c in df.columns:
        if pd.api.types.is_datetime64_any_dtype(df[c]):
            return c
    return None


def extract_daily_features(df, sensor_name, ts_col=None, subject_col='subject_id'):
    """
    센서 데이터프레임 → 하루 단위 집계 피처 반환

    집계 통계: mean, std, min, max, count
    """
    df = df.copy()

    # 타임스탬프 처리
    if ts_col is None:
        ts_col = get_timestamp_col(df)

    if ts_col and ts_col in df.columns:
        df[ts_col] = pd.to_datetime(df[ts_col], unit='ms', errors='coerce') \
                     if df[ts_col].dtype in ['int64','float64'] \
                     else pd.to_datetime(df[ts_col], errors='coerce')
        df['date'] = df[ts_col].dt.date
    elif 'date' not in df.columns:
        print(f'  ⚠️  [{sensor_name}] 타임스탬프 컬럼을 찾지 못했습니다.')
        return None

    df['date'] = pd.to_datetime(df['date'])

    # 수치형 컬럼만 선택 (subject_id, date 제외)
    exclude = {subject_col, 'date', ts_col}
    num_cols = [c for c in df.select_dtypes(include=[np.number]).columns
                if c not in exclude]

    if not num_cols:
        print(f'  ⚠️  [{sensor_name}] 수치형 컬럼이 없습니다.')
        return None

    # 하루 단위 집계
    grp = df.groupby([subject_col, 'date'])[num_cols]
    aggs = {
        'mean' : grp.mean(),
        'std'  : grp.std().fillna(0),
        'min'  : grp.min(),
        'max'  : grp.max(),
        'count': grp.count(),
    }

    # 멀티-집계 컬럼 병합
    result_parts = []
    for agg_name, agg_df in aggs.items():
        agg_df.columns = [f'{sensor_name}__{c}__{agg_name}' for c in agg_df.columns]
        result_parts.append(agg_df)

    result = pd.concat(result_parts, axis=1).reset_index()
    print(f'  ✅ [{sensor_name}]: {result.shape[1]-2}개 피처, {result.shape[0]}행')
    return result


print('🔧 피처 엔지니어링 함수 정의 완료')

🔧 피처 엔지니어링 함수 정의 완료


In [ ]:
# ─── 모든 센서 → 일별 피처 추출 ──────────────────────────
print('📊 일별 피처 추출 시작...')

daily_features = {}
for key, df in raw.items():
    feat = extract_daily_features(df, sensor_name=key)
    if feat is not None:
        daily_features[key] = feat

print(f'\n✅ 피처 추출 완료: {len(daily_features)}개 센서')

📊 일별 피처 추출 시작...
  ✅ [ac_status]: 5개 피처, 700행
  ✅ [activity]: 5개 피처, 700행
  ⚠️  [ambience] 수치형 컬럼이 없습니다.
  ⚠️  [ble] 수치형 컬럼이 없습니다.
  ⚠️  [gps] 수치형 컬럼이 없습니다.
  ✅ [light]: 5개 피처, 700행
  ✅ [screen]: 5개 피처, 700행
  ⚠️  [usage_stats] 수치형 컬럼이 없습니다.
  ⚠️  [wifi] 수치형 컬럼이 없습니다.
  ⚠️  [hr] 수치형 컬럼이 없습니다.
  ✅ [w_light]: 5개 피처, 664행
  ✅ [pedo]: 35개 피처, 653행

✅ 피처 추출 완료: 6개 센서


모든 "수치형 없음" 센서의 공통 원인: 값이 중첩 list/struct 컬럼으로 저장되어 있어서 select_dtypes가 탐지 못함
| 센서 | 실제 컬럼 구조 |
| --- | --- |
| `hr` | `heart_rate`: `List[int]` — HR 값 배열 |
| `wifi` | `m_wifi`: `List[{bssid:str, rssi:int}]` |
| `usage_stats` | `m_usage_stats`: `List[{app_name:str, total_time:int}]` |
| `ble` | `m_ble`: `List[{address:str, device_class:str, rssi:int}]` |
| `gps` | `m_gps`: `List[{altitude,latitude,longitude,speed}]` |
| `ambience` | `m_ambience`: `List[List[str]]` (이중 중첩) |

In [ ]:
# ─── 중첩 리스트 컬럼 → 일별 피처 추출 함수들 ────────────

def extract_hr_features_nested(df, subject_col='subject_id'):
    """heart_rate: List[int] → 일별 통계"""
    df = df.copy()
    df['timestamp'] = pd.to_datetime(df['timestamp'], unit='ms', errors='coerce')
    df['date'] = df['timestamp'].dt.date
    df['hour'] = df['timestamp'].dt.hour
    df['date'] = pd.to_datetime(df['date'])

    # list explode → 각 행이 HR 1개 값
    df = df.explode('heart_rate')
    df['heart_rate'] = pd.to_numeric(df['heart_rate'], errors='coerce')
    df = df.dropna(subset=['heart_rate'])
    df = df[df['heart_rate'].between(30, 220)]  # 생리적 범위 필터

    results = []
    for (subj, date), grp in df.groupby([subject_col, 'date']):
        hr = grp['heart_rate']
        night = grp[grp['hour'].between(0, 6)]['heart_rate']
        pre_sleep = grp[grp['hour'].between(21, 23)]['heart_rate']

        results.append({
            subject_col: subj, 'date': date,
            # 전일 HR 통계
            'hr__mean':    hr.mean(),
            'hr__std':     hr.std(ddof=0),
            'hr__min':     hr.min(),
            'hr__max':     hr.max(),
            'hr__count':   len(hr),
            'hr__resting': hr.quantile(0.1),          # 안정시 HR
            'hr__rmssd':   float(np.sqrt(np.mean(np.diff(hr.values)**2)))
                           if len(hr) > 1 else 0,     # HRV 근사
            # 야간 HR (수면 품질 직접 지표)
            'hr__night_mean': night.mean() if len(night) > 0 else np.nan,
            'hr__night_std':  night.std(ddof=0) if len(night) > 1 else 0,
            'hr__night_min':  night.min() if len(night) > 0 else np.nan,
            # 취침 전 HR (각성 상태)
            'hr__presleep_mean': pre_sleep.mean() if len(pre_sleep) > 0 else np.nan,
        })

    result_df = pd.DataFrame(results)
    print(f'  ✅ [hr]: {result_df.shape[1]-2}개 피처, {len(result_df)}행')
    return result_df


def extract_wifi_features_nested(df, subject_col='subject_id'):
    """m_wifi: List[{bssid:str, rssi:int}] → 이동성·귀가 피처"""
    df = df.copy()
    df['timestamp'] = pd.to_datetime(df['timestamp'], unit='ms', errors='coerce')
    df['date'] = df['timestamp'].dt.date
    df['hour'] = df['timestamp'].dt.hour
    df['date'] = pd.to_datetime(df['date'])

    # list explode → 각 행이 dict 1개
    df = df.explode('m_wifi').dropna(subset=['m_wifi'])
    wifi_expanded = pd.json_normalize(df['m_wifi'].tolist())
    df = pd.concat([
        df[[subject_col, 'date', 'hour', 'timestamp']].reset_index(drop=True),
        wifi_expanded.reset_index(drop=True)
    ], axis=1)

    # rssi 수치 변환
    if 'rssi' in df.columns:
        df['rssi'] = pd.to_numeric(df['rssi'], errors='coerce')

    # 피험자별 홈 네트워크 추정 (가장 자주 등장하는 bssid)
    home_bssid = (df.groupby(subject_col)['bssid']
                    .agg(lambda x: x.value_counts().index[0])
                    .to_dict()) if 'bssid' in df.columns else {}
    if home_bssid:
        df['is_home'] = df.apply(
            lambda r: int(r['bssid'] == home_bssid.get(r[subject_col], '')), axis=1)

    results = []
    for (subj, date), grp in df.groupby([subject_col, 'date']):
        night_grp = grp[grp['hour'].between(0, 6)]

        row = {
            subject_col: subj, 'date': date,
            # 이동성 지표
            'wifi__scan_count':      len(grp),
            'wifi__unique_bssid':    grp['bssid'].nunique() if 'bssid' in grp else 0,
            # 야간 스캔 (수면 중 폰 사용 proxy)
            'wifi__night_scan_count': len(night_grp),
            # 귀가/재실 지표
            'wifi__home_ratio':  grp['is_home'].mean() if 'is_home' in grp else 0,
            'wifi__home_count':  grp['is_home'].sum()  if 'is_home' in grp else 0,
        }
        # 신호 강도 통계
        if 'rssi' in grp.columns and grp['rssi'].notna().sum() > 0:
            row['wifi__rssi_mean'] = grp['rssi'].mean()
            row['wifi__rssi_std']  = grp['rssi'].std(ddof=0)
            row['wifi__rssi_home_mean'] = (grp[grp['is_home']==1]['rssi'].mean()
                                           if 'is_home' in grp and grp['is_home'].sum() > 0
                                           else np.nan)
        # 귀가 시각 (홈 WiFi 첫 등장 시간, 18시 이후)
        if 'is_home' in grp.columns:
            evening_home = grp[(grp['is_home']==1) & (grp['hour'] >= 18)]
            row['wifi__home_arrival_hour'] = (evening_home['hour'].min()
                                              if len(evening_home) > 0 else np.nan)
        results.append(row)

    result_df = pd.DataFrame(results)
    print(f'  ✅ [wifi]: {result_df.shape[1]-2}개 피처, {len(result_df)}행')
    return result_df


def extract_usage_stats_features_nested(df, subject_col='subject_id'):
    """m_usage_stats: List[{app_name:str, total_time:int}] → 앱 사용 피처"""
    df = df.copy()
    df['timestamp'] = pd.to_datetime(df['timestamp'], unit='ms', errors='coerce')
    df['date'] = df['timestamp'].dt.date
    df['hour'] = df['timestamp'].dt.hour
    df['date'] = pd.to_datetime(df['date'])

    # list explode → dict 확장
    df = df.explode('m_usage_stats').dropna(subset=['m_usage_stats'])
    usage_expanded = pd.json_normalize(df['m_usage_stats'].tolist())
    df = pd.concat([
        df[[subject_col, 'date', 'hour']].reset_index(drop=True),
        usage_expanded.reset_index(drop=True)
    ], axis=1)

    if 'total_time' in df.columns:
        df['total_time'] = pd.to_numeric(df['total_time'], errors='coerce').fillna(0)
        df['total_time_min'] = df['total_time'] / 60000  # ms → 분

    # 앱 카테고리 분류
    CATEGORIES = {
        'social':  ['kakao', 'instagram', 'facebook', 'twitter', 'tiktok',
                    'snapchat', 'line', 'telegram', 'whatsapp', 'naver.band'],
        'video':   ['youtube', 'netflix', 'tving', 'watcha', 'wavve',
                    'twitch', 'video', 'vlive'],
        'game':    ['game', 'com.nexon', 'com.netmarble', 'pubg', 'minecraft'],
        'browser': ['chrome', 'samsung.internet', 'firefox', 'naver', 'browser'],
        'work':    ['office', 'notion', 'slack', 'zoom', 'teams',
                    'gmail', 'outlook', 'calendar', 'docs'],
    }
    if 'app_name' in df.columns:
        df['category'] = 'other'
        for cat, kws in CATEGORIES.items():
            mask = df['app_name'].str.lower().str.contains('|'.join(kws), na=False)
            df.loc[mask, 'category'] = cat

    results = []
    for (subj, date), grp in df.groupby([subject_col, 'date']):
        row = {subject_col: subj, 'date': date,
               'usage__app_count': grp['app_name'].nunique() if 'app_name' in grp else 0}

        if 'total_time_min' in grp.columns:
            total = grp['total_time_min'].sum()
            row['usage__total_min'] = total

            # 취침 전 2시간 (22~23시) 사용량 — SOL과 직결
            evening = grp[grp['hour'].between(22, 23)]['total_time_min'].sum()
            row['usage__evening_min']   = evening
            row['usage__evening_ratio'] = evening / (total + 1e-6)

            # 카테고리별 사용량
            for cat in CATEGORIES:
                t = grp[grp['category']==cat]['total_time_min'].sum()
                row[f'usage__{cat}_min']   = t
                row[f'usage__{cat}_ratio'] = t / (total + 1e-6)

        results.append(row)

    result_df = pd.DataFrame(results)
    print(f'  ✅ [usage_stats]: {result_df.shape[1]-2}개 피처, {len(result_df)}행')
    return result_df


def extract_ble_features_nested(df, subject_col='subject_id'):
    """m_ble: List[{address:str, device_class:str, rssi:int}] → 사회적 근접성"""
    df = df.copy()
    df['timestamp'] = pd.to_datetime(df['timestamp'], unit='ms', errors='coerce')
    df['date'] = df['timestamp'].dt.date
    df['hour'] = df['timestamp'].dt.hour
    df['date'] = pd.to_datetime(df['date'])

    df = df.explode('m_ble').dropna(subset=['m_ble'])
    ble_exp = pd.json_normalize(df['m_ble'].tolist())
    df = pd.concat([df[[subject_col,'date','hour']].reset_index(drop=True),
                    ble_exp.reset_index(drop=True)], axis=1)

    if 'rssi' in df.columns:
        df['rssi'] = pd.to_numeric(df['rssi'], errors='coerce')

    results = []
    for (subj, date), grp in df.groupby([subject_col, 'date']):
        night = grp[grp['hour'].between(0, 6)]
        row = {
            subject_col: subj, 'date': date,
            'ble__scan_count':        len(grp),
            'ble__unique_devices':    grp['address'].nunique() if 'address' in grp else 0,
            'ble__night_scan_count':  len(night),
            'ble__night_unique_devs': night['address'].nunique()
                                      if 'address' in night.columns and len(night) > 0 else 0,
        }
        if 'rssi' in grp.columns and grp['rssi'].notna().sum() > 0:
            row['ble__rssi_mean'] = grp['rssi'].mean()
            row['ble__rssi_max']  = grp['rssi'].max()  # 가장 가까운 기기
        results.append(row)

    result_df = pd.DataFrame(results)
    print(f'  ✅ [ble]: {result_df.shape[1]-2}개 피처, {len(result_df)}행')
    return result_df


def extract_gps_features_nested(df, subject_col='subject_id'):
    """m_gps: List[{altitude,latitude,longitude,speed}] → 이동 범위·속도"""
    df = df.copy()
    df['timestamp'] = pd.to_datetime(df['timestamp'], unit='ms', errors='coerce')
    df['date'] = df['timestamp'].dt.date
    df['hour'] = df['timestamp'].dt.hour
    df['date'] = pd.to_datetime(df['date'])

    df = df.explode('m_gps').dropna(subset=['m_gps'])
    gps_exp = pd.json_normalize(df['m_gps'].tolist())
    df = pd.concat([df[[subject_col,'date','hour']].reset_index(drop=True),
                    gps_exp.reset_index(drop=True)], axis=1)

    for c in ['latitude','longitude','speed','altitude']:
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors='coerce')

    results = []
    for (subj, date), grp in df.groupby([subject_col, 'date']):
        row = {subject_col: subj, 'date': date,
               'gps__fix_count': len(grp)}
        if 'latitude' in grp.columns and grp['latitude'].notna().sum() > 1:
            # 이동 반경 (위·경도 표준편차로 근사)
            row['gps__lat_std'] = grp['latitude'].std()
            row['gps__lon_std'] = grp['longitude'].std()
            row['gps__radius']  = np.sqrt(grp['latitude'].std()**2 +
                                          grp['longitude'].std()**2)
        if 'speed' in grp.columns and grp['speed'].notna().sum() > 0:
            row['gps__speed_mean'] = grp['speed'].mean()
            row['gps__speed_max']  = grp['speed'].max()
            row['gps__moving_ratio'] = (grp['speed'] > 0.5).mean()  # 이동 시간 비율
        results.append(row)

    result_df = pd.DataFrame(results)
    print(f'  ✅ [gps]: {result_df.shape[1]-2}개 피처, {len(result_df)}행')
    return result_df


def extract_ambience_features_nested(df, subject_col='subject_id'):
    """m_ambience: List[List[str]] → 소리 환경 분포 (이중 중첩)"""
    df = df.copy()
    df['timestamp'] = pd.to_datetime(df['timestamp'], unit='ms', errors='coerce')
    df['date'] = df['timestamp'].dt.date
    df['date'] = pd.to_datetime(df['date'])

    # 이중 중첩 해제: 각 행의 리스트들을 flat하게 만들기
    def flatten_ambience(val):
        if val is None:
            return []
        if isinstance(val, list):
            result = []
            for item in val:
                if isinstance(item, list):
                    result.extend(item)
                else:
                    result.append(str(item))
            return result
        return [str(val)]

    df['labels'] = df['m_ambience'].apply(flatten_ambience)
    df = df.explode('labels').dropna(subset=['labels'])
    df['labels'] = df['labels'].astype(str).str.strip().str.lower()
    df = df[df['labels'] != '']

    results = []
    for (subj, date), grp in df.groupby([subject_col, 'date']):
        total = len(grp)
        vc = grp['labels'].value_counts()
        row = {
            subject_col: subj, 'date': date,
            'ambience__total_count': total,
            'ambience__unique_labels': grp['labels'].nunique(),
        }
        # 상위 레이블 비율 피처 (실제 레이블 확인 후 추가 가능)
        for lbl in vc.index[:10]:
            safe = lbl.replace(' ', '_').replace('/', '_')
            row[f'ambience__{safe}_ratio'] = vc[lbl] / total
        results.append(row)

    result_df = pd.DataFrame(results)
    print(f'  ✅ [ambience]: {result_df.shape[1]-2}개 피처, {len(result_df)}행')
    return result_df

In [ ]:
# ─── 중첩 센서 피처 추출 실행 ──────────────────────────────
print('📊 중첩 리스트 센서 피처 추출...')

NESTED_EXTRACTORS = {
    'hr':          extract_hr_features_nested,
    'wifi':        extract_wifi_features_nested,
    'usage_stats': extract_usage_stats_features_nested,
    'ble':         extract_ble_features_nested,
    'gps':         extract_gps_features_nested,
    'ambience':    extract_ambience_features_nested,
}

for key, func in NESTED_EXTRACTORS.items():
    if key in raw:
        feat = func(raw[key])
        if feat is not None:
            daily_features[key] = feat

print(f'\n✅ 최종 피처 딕셔너리: {list(daily_features.keys())}')

📊 중첩 리스트 센서 피처 추출...
  ✅ [hr]: 11개 피처, 636행
  ✅ [wifi]: 9개 피처, 685행
  ✅ [usage_stats]: 14개 피처, 690행
  ✅ [ble]: 6개 피처, 651행
  ✅ [gps]: 7개 피처, 660행
  ✅ [ambience]: 4986개 피처, 700행

✅ 최종 피처 딕셔너리: ['ac_status', 'activity', 'light', 'screen', 'w_light', 'pedo', 'hr', 'wifi', 'usage_stats', 'ble', 'gps', 'ambience']


## HR 센서에서 추출 가능한 핵심 피처

### 실제 수면 구간 추정

HR은 수면 중 **10~20 bpm 감소**하고 유지됨. 이걸 이용해 수면 시작·종료 시각을 직접 추정할 수 있음.

깨어있을 때: HR ~75 bpm
수면 시작:   HR이 임계값 아래로 떨어지고 15분+ 유지
수면 중:     HR ~60 bpm (안정)
기상:        HR이 다시 상승

| 피처 | S/Q 레이블 연결 |
| --- | --- |
| 추정 수면 시작 시각 | S3(SOL), Q1 |
| 추정 기상 시각 | S1(TST) |
| 추정 총 수면 시간 | S1(TST) |
| 수면 중 HR 평균/std | S2(SE), Q1 |
| 취침 전→수면 HR 감소폭 | Q3(스트레스) |
| 수면 중 HR 상승 이벤트 수 | S4(WASO) |

## WiFi 센서에서 추출 가능한 핵심 피처

### 귀가, 취침 시각 추정

야간 스캔 끊김 시각  → 폰 내려놓은 시각 ≈ 취침 시각
홈 WiFi 첫 연결 시각 → 귀가 시각
아침 첫 스캔 시각    → 기상 후 폰 집어든 시각

## Usage Status에서 추출 가능한 핵심 피처

### 실제 취침, 기상 시각 추정

마지막 앱 사용 시각  → 취침 직전 시각 (가장 정확한 SOL 예측 변수)
첫 아침 앱 사용 시각 → 기상 시각
수면 시간대 앱 사용  → WASO(각성) 이벤트

---

## CODE

### 각 함수가 만드는 칼럼 정리

### `estimate_sleep_from_hr` → 8개

| 컬럼 | 의미 | 연결 레이블 |
| --- | --- | --- |
| `hr__sleep_onset_hour` | 추정 수면 시작 시각 (e.g. 23.5 = 23:30) | S3(SOL) |
| `hr__sleep_offset_hour` | 추정 기상 시각 | S1(TST) |
| `hr__est_tst_min` | 추정 총 수면 시간 (분) | S1(TST) |
| `hr__sleep_hr_mean` | 수면 중 평균 HR | S2(SE), Q1 |
| `hr__sleep_hr_std` | 수면 중 HR 변동성 | Q1(수면질) |
| `hr__sleep_hr_min` | 수면 중 최저 HR (깊은 수면 proxy) | Q1(수면질) |
| `hr__presleep_hr_drop` | 취침 전 HR - 수면 중 HR (클수록 잘 이완됨) | Q3(스트레스) |
| `hr__arousal_count` | 수면 중 HR이 임계값 초과한 횟수 (각성 이벤트) | S4(WASO) |

### `extract_wifi_bedtime_proxy` → 3개

| 컬럼 | 의미 | 연결 레이블 |
| --- | --- | --- |
| `wifi__phone_down_hour` | WiFi 스캔이 30분+ 끊긴 시점 (폰 내려놓은 시각) | S3(SOL) |
| `wifi__last_night_scan_hour` | 야간 마지막 WiFi 스캔 시각 | S3(SOL) |
| `wifi__first_morning_scan_hour` | 아침 첫 WiFi 스캔 시각 (기상 proxy) | S1(TST) |

### `extract_usage_bedtime_proxy` → 4개
| 컬럼 | 의미 | 연결 레이블 |
| --- | --- | --- |
| `usage__last_use_hour` | 저녁(20시~) 마지막 앱 사용 시각 | S3(SOL), Q2 |
| `usage__first_morning_hour` | 아침 첫 앱 사용 시각 | S1(TST) |
| `usage__sleep_hour_count` | 0~6시 앱 사용 횟수 (수면 중 각성) | S4(WASO) |
| `usage__phone_off_duration_hr` | 마지막 앱 ~ 첫 아침 앱까지 공백 시간 | S1(TST) |

In [ ]:
# ─── HR 기반 수면 구간 추정 ────────────────────────────────
def estimate_sleep_from_hr(df, subject_col='subject_id',
                            min_sleep_duration_min=60,
                            low_hr_window_min=15):
    """
    HR이 개인 임계값 아래로 sustained 하게 떨어지는 구간 = 수면 구간

    반환: 피험자×날짜별 추정 수면 시작/종료/길이 피처
    """
    df = df.copy()
    df['timestamp'] = pd.to_datetime(df['timestamp'], unit='ms', errors='coerce')
    df = df.explode('heart_rate')
    df['heart_rate'] = pd.to_numeric(df['heart_rate'], errors='coerce')
    df = df.dropna(subset=['heart_rate', 'timestamp'])
    df = df[df['heart_rate'].between(30, 220)]
    df = df.sort_values([subject_col, 'timestamp'])

    results = []

    for subj, subj_df in df.groupby(subject_col):
        # 개인 임계값: 전체 HR 하위 35% → "수면 HR" 기준
        sleep_threshold = subj_df['heart_rate'].quantile(0.35)

        # 날짜 기준: 전날 20시 ~ 당일 12시 (수면 윈도우)
        dates = subj_df['timestamp'].dt.date.unique()

        for date in dates:
            date = pd.Timestamp(date)
            window_start = date - pd.Timedelta(hours=4)   # 전날 20시
            window_end   = date + pd.Timedelta(hours=12)  # 당일 12시

            window = subj_df[
                (subj_df['timestamp'] >= window_start) &
                (subj_df['timestamp'] <= window_end)
            ].copy()

            if len(window) < 10:
                continue

            # 1분 단위 리샘플 후 rolling mean으로 노이즈 제거
            window = window.set_index('timestamp')['heart_rate']
            window_1min = window.resample('1min').mean().interpolate()

            # 임계값 아래 구간 탐지
            is_low = window_1min < sleep_threshold

            # 연속 구간 찾기
            sleep_onset = None
            sleep_offset = None
            max_duration = 0

            in_sleep = False
            seg_start = None

            for t, low in is_low.items():
                if low and not in_sleep:
                    in_sleep = True
                    seg_start = t
                elif not low and in_sleep:
                    duration = (t - seg_start).total_seconds() / 60
                    if duration > max_duration and duration >= min_sleep_duration_min:
                        max_duration = duration
                        sleep_onset  = seg_start
                        sleep_offset = t
                    in_sleep = False

            # 수면 구간 피처 계산
            row = {subject_col: subj, 'date': date}

            if sleep_onset and sleep_offset:
                sleep_hr = window_1min[sleep_onset:sleep_offset]
                pre_sleep_hr = window_1min[
                    max(window_1min.index[0], sleep_onset - pd.Timedelta(hours=1))
                    :sleep_onset
                ]

                row.update({
                    'hr__sleep_onset_hour':  sleep_onset.hour + sleep_onset.minute/60,
                    'hr__sleep_offset_hour': sleep_offset.hour + sleep_offset.minute/60,
                    'hr__est_tst_min':       max_duration,               # 추정 TST
                    'hr__sleep_hr_mean':     sleep_hr.mean(),
                    'hr__sleep_hr_std':      sleep_hr.std(),
                    'hr__sleep_hr_min':      sleep_hr.min(),              # 가장 깊은 수면 HR
                    # 취침 전 대비 수면 HR 감소폭 (클수록 회복 수면)
                    'hr__presleep_hr_drop':  (pre_sleep_hr.mean() - sleep_hr.mean()
                                              if len(pre_sleep_hr) > 0 else np.nan),
                    # 수면 중 HR 상승 이벤트 (각성 proxy → S4 WASO)
                    'hr__arousal_count':     int((sleep_hr > sleep_threshold).sum()),
                })
            else:
                # 수면 감지 실패 → NaN
                for col in ['hr__sleep_onset_hour','hr__sleep_offset_hour',
                            'hr__est_tst_min','hr__sleep_hr_mean','hr__sleep_hr_std',
                            'hr__sleep_hr_min','hr__presleep_hr_drop','hr__arousal_count']:
                    row[col] = np.nan

            results.append(row)

    result_df = pd.DataFrame(results)
    print(f'  ✅ [hr_sleep]: {result_df.shape[1]-2}개 피처, {len(result_df)}행')
    return result_df


# ─── WiFi 취침 시각 프록시 ────────────────────────────────
def extract_wifi_bedtime_proxy(df, subject_col='subject_id'):
    """
    야간 WiFi 스캔이 끊기는 시각 → 폰 내려놓은 시각 추정
    """
    df = df.copy()
    df['timestamp'] = pd.to_datetime(df['timestamp'], unit='ms', errors='coerce')
    df['date'] = df['timestamp'].dt.date
    df['date'] = pd.to_datetime(df['date'])
    df['hour'] = df['timestamp'].dt.hour

    results = []
    for (subj, date), grp in df.groupby([subject_col, 'date']):
        # 야간 스캔 (20시~다음날 6시)
        night = grp[grp['hour'].between(20, 23) | grp['hour'].between(0, 6)]
        night = night.sort_values('timestamp')

        row = {subject_col: subj, 'date': date}

        if len(night) >= 2:
            # 스캔 간격이 갑자기 길어지는 시점 = 폰 내려놓은 시각
            night = night.set_index('timestamp')
            gaps = night.index.to_series().diff().dt.total_seconds() / 60  # 분

            # 30분 이상 공백이 처음 생기는 시점
            long_gap = gaps[gaps > 30]
            if len(long_gap) > 0:
                gap_start = long_gap.index[0] - pd.Timedelta(minutes=gaps[long_gap.index[0]])
                row['wifi__phone_down_hour'] = gap_start.hour + gap_start.minute/60
            else:
                row['wifi__phone_down_hour'] = np.nan

            # 마지막 야간 스캔 시각
            last_scan = night.index[-1]
            row['wifi__last_night_scan_hour'] = last_scan.hour + last_scan.minute/60

            # 첫 아침 스캔 (5~9시)
            morning = grp[grp['hour'].between(5, 9)].sort_values('timestamp')
            row['wifi__first_morning_scan_hour'] = (
                morning['timestamp'].iloc[0].hour + morning['timestamp'].iloc[0].minute/60
                if len(morning) > 0 else np.nan
            )
        else:
            row.update({'wifi__phone_down_hour': np.nan,
                        'wifi__last_night_scan_hour': np.nan,
                        'wifi__first_morning_scan_hour': np.nan})

        results.append(row)

    result_df = pd.DataFrame(results)
    print(f'  ✅ [wifi_bedtime]: {result_df.shape[1]-2}개 피처, {len(result_df)}행')
    return result_df


# ─── Usage Stats 취침·기상 시각 프록시 ───────────────────
def extract_usage_bedtime_proxy(df, subject_col='subject_id'):
    """
    마지막 앱 사용 시각 → 취침 시각 / 첫 아침 사용 → 기상 시각
    """
    df = df.copy()
    df['timestamp'] = pd.to_datetime(df['timestamp'], unit='ms', errors='coerce')
    df['date'] = df['timestamp'].dt.date
    df['date'] = pd.to_datetime(df['date'])
    df['hour'] = df['timestamp'].dt.hour

    results = []
    for (subj, date), grp in df.groupby([subject_col, 'date']):
        row = {subject_col: subj, 'date': date}

        # 마지막 앱 사용 시각 (20시 이후)
        evening = grp[grp['hour'] >= 20].sort_values('timestamp')
        if len(evening) > 0:
            last = evening['timestamp'].iloc[-1]
            row['usage__last_use_hour'] = last.hour + last.minute/60
        else:
            row['usage__last_use_hour'] = np.nan

        # 첫 아침 앱 사용 시각 (5~10시)
        morning = grp[grp['hour'].between(5, 10)].sort_values('timestamp')
        if len(morning) > 0:
            first = morning['timestamp'].iloc[0]
            row['usage__first_morning_hour'] = first.hour + first.minute/60
        else:
            row['usage__first_morning_hour'] = np.nan

        # 수면 시간대(0~6시) 앱 사용 횟수 → WASO proxy
        row['usage__sleep_hour_count'] = (grp['hour'].between(0, 6)).sum()

        # 취침~기상 추정 공백 시간
        if not pd.isna(row.get('usage__last_use_hour')) and \
           not pd.isna(row.get('usage__first_morning_hour')):
            onset = row['usage__last_use_hour']
            offset = row['usage__first_morning_hour']
            # 자정 넘김 처리
            gap = (offset - onset) if offset > onset else (24 - onset + offset)
            row['usage__phone_off_duration_hr'] = gap  # 폰 미사용 시간 ≈ 수면 시간

        results.append(row)

    result_df = pd.DataFrame(results)
    print(f'  ✅ [usage_bedtime]: {result_df.shape[1]-2}개 피처, {len(result_df)}행')
    return result_df

In [ ]:
# ─── 실행 (중첩 센서 추출 이후에 추가) ───────────────────
print('📊 수면 시각 추정 피처 추출...')

if 'hr' in raw:
    hr_sleep = estimate_sleep_from_hr(raw['hr'])
    # 기존 hr 피처와 병합
    if 'hr' in daily_features:
        daily_features['hr'] = pd.merge(
            daily_features['hr'], hr_sleep, on=['subject_id','date'], how='outer')
    else:
        daily_features['hr_sleep'] = hr_sleep

if 'wifi' in raw:
    wifi_bed = extract_wifi_bedtime_proxy(raw['wifi'])
    if 'wifi' in daily_features:
        daily_features['wifi'] = pd.merge(
            daily_features['wifi'], wifi_bed, on=['subject_id','date'], how='outer')

if 'usage_stats' in raw:
    usage_bed = extract_usage_bedtime_proxy(raw['usage_stats'])
    if 'usage_stats' in daily_features:
        daily_features['usage_stats'] = pd.merge(
            daily_features['usage_stats'], usage_bed, on=['subject_id','date'], how='outer')

📊 수면 시각 추정 피처 추출...
  ✅ [hr_sleep]: 8개 피처, 588행
  ✅ [wifi_bedtime]: 3개 피처, 685행
  ✅ [usage_bedtime]: 4개 피처, 690행
